In [14]:
import numpy as np
import sounddevice as sd
import soundfile as sf
import librosa

import torch
import torch.nn as nn

In [15]:
# ============================================
# Configuration
# ============================================

SAMPLE_RATE = 16000

FRAME_LENGTH = 0.025
FRAME_HOP = 0.020

N_FFT = 400
HOP_LENGTH = 320
N_MELS = 80

CHUNK_DURATION = 1.0
CHUNK_SAMPLES = int(
    SAMPLE_RATE * CHUNK_DURATION
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [16]:
# ============================================
# Load phoneme vocabulary
# ============================================

# import glob
# import pyarrow.parquet as pq

# parquet_files = sorted(
#     glob.glob(
#         "../data/raw/librispeech-segment/"
#         "train.clean.100-*.parquet"
#     )
# )

# phone_counts = {}

# for path in parquet_files:

#     table = pq.read_table(
#         path,
#         columns=["phones"]
#     )

#     phone_lists = (
#         table["phones"].to_pylist()
#     )

#     for phone_list in phone_lists:

#         for phone in phone_list:

#             phone_counts[phone] = (
#                 phone_counts.get(phone, 0)
#                 + 1
#             )

# phones = sorted(
#     phone_counts.keys()
# )

phones = ['AA', 'AE', 'AH', 'AO', 'AW', 'AY', 'B', 'CH', 'D', 'DH', 'EH', 'ER', 'EY', 'F', 'G', 'HH', 'IH', 'IY', 'JH', 'K', 'L', 'M', 'N', 'NG', 'OW', 'OY', 'P', 'R', 'S', 'SH', 'T', 'TH', 'UH', 'UW', 'V', 'W', 'Y', 'Z', 'ZH', '[SIL]', '[UNK]']

phone_to_id = {
    phone: i
    for i, phone in enumerate(phones)
}

id_to_phone = {
    i: phone
    for phone, i in phone_to_id.items()
}

NUM_CLASSES = len(phones)

print(
    "Number of classes:",
    NUM_CLASSES
)

print(phones)

Number of classes: 41
['AA', 'AE', 'AH', 'AO', 'AW', 'AY', 'B', 'CH', 'D', 'DH', 'EH', 'ER', 'EY', 'F', 'G', 'HH', 'IH', 'IY', 'JH', 'K', 'L', 'M', 'N', 'NG', 'OW', 'OY', 'P', 'R', 'S', 'SH', 'T', 'TH', 'UH', 'UW', 'V', 'W', 'Y', 'Z', 'ZH', '[SIL]', '[UNK]']


In [17]:
# ============================================
# CNN-LSTM model
# ============================================

class CNNLSTM(nn.Module):

    def __init__(
        self,
        num_classes,
        hidden_size=128,
    ):
        super().__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm2d(32),

            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=(2, 1)
            ),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm2d(64),

            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=(2, 1)
            ),

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm2d(128),

            nn.ReLU(),
        )

        cnn_feature_size = 128 * 20

        self.lstm = nn.LSTM(
            input_size=cnn_feature_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )

        # dropoout at 0.4
        self.dropout = nn.Dropout(
            p=0.4
        )

        self.classifier = nn.Linear(
            hidden_size,
            num_classes,
        )

    def forward(self, x):

        x = x.unsqueeze(1)

        x = self.cnn(x)

        x = x.permute(
            0, 3, 1, 2
        )

        x = x.flatten(
            start_dim=2
        )

        x, _ = self.lstm(x)

        x = self.dropout(x)

        x = self.classifier(x)

        return x

In [18]:
# ============================================
# Load trained model
# ============================================

model = CNNLSTM(
    num_classes=NUM_CLASSES,
    hidden_size=128,
)

model.load_state_dict(
    torch.load(
        "../checkpoints/best_cnn_lstm_128_dropout_04_cnn_3.pt",
        map_location=DEVICE
    )
)

model.to(DEVICE)
model.eval()

print("model loaded.")

model loaded.


In [19]:
# ============================================
# Audio → Log-Mel
# ============================================

def audio_to_log_mel(audio):

    # Ensure exactly 1 second

    if len(audio) < CHUNK_SAMPLES:

        audio = np.pad(
            audio,
            (
                0,
                CHUNK_SAMPLES - len(audio)
            )
        )

    else:

        audio = audio[
            :CHUNK_SAMPLES
        ]

    log_mel = (
        librosa.feature.melspectrogram(
            y=audio,
            sr=SAMPLE_RATE,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            power=2.0,
            center=False,
        )
    )

    log_mel = librosa.power_to_db(
        log_mel,
        ref=np.max
    )

    return log_mel.astype(
        np.float32
    )

In [20]:
# ============================================
# Predict full phoneme sequence
# ============================================

def predict_phonemes(
    audio_buffer
):

    log_mel = audio_to_log_mel(
        audio_buffer
    )

    X = torch.from_numpy(
        log_mel
    ).float().unsqueeze(0).to(
        DEVICE
    )

    with torch.inference_mode():

        output = model(X)

        probabilities = torch.softmax(
            output,
            dim=-1
        )

        confidence, prediction = (
            probabilities.max(dim=-1)
        )

    prediction_ids = (
        prediction[0]
        .cpu()
        .numpy()
    )

    confidence_values = (
        confidence[0]
        .cpu()
        .numpy()
    )

    phoneme_labels = [
        id_to_phone[int(i)]
        for i in prediction_ids
    ]

    return (
        phoneme_labels,
        confidence_values
    )

In [21]:
# ============================================
# Microphone test
# ============================================

RECORD_SECONDS = 3

print("Recording...")

audio = sd.rec(
    int(
        RECORD_SECONDS
        * SAMPLE_RATE
    ),
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype="float32",
)

sd.wait()

audio = audio.squeeze()

print(
    "Recorded samples:",
    len(audio)
)

print(
    "Duration:",
    len(audio) / SAMPLE_RATE,
    "seconds"
)

Recording...


Recorded samples: 48000
Duration: 3.0 seconds


In [22]:
# ============================================
# Real-time settings
# ============================================

AUDIO_BLOCK_MS = 20

AUDIO_BLOCK_SAMPLES = int(
    SAMPLE_RATE
    * AUDIO_BLOCK_MS
    / 1000
)

# Run model every 40 ms
INFERENCE_INTERVAL_MS = 40

INFERENCE_BLOCKS = (
    INFERENCE_INTERVAL_MS
    // AUDIO_BLOCK_MS
)

# Number of model predictions produced
# during each inference interval
OUTPUT_FRAMES = INFERENCE_BLOCKS

print(
    "Audio block:",
    AUDIO_BLOCK_MS,
    "ms"
)

print(
    "Inference interval:",
    INFERENCE_INTERVAL_MS,
    "ms"
)

print(
    "Output frames per inference:",
    OUTPUT_FRAMES
)

Audio block: 20 ms
Inference interval: 40 ms
Output frames per inference: 2


In [23]:
import queue

audio_queue = queue.Queue()


def audio_callback(indata, frames, time_info, status):
    if status:
        print(status)

    audio_queue.put(
        indata[:, 0].copy()
    )

In [24]:
arpabet_to_visemes = {
    # Open / wide
    "AA": "OPEN",
    "AE": "OPEN",
    "AH": "OPEN",
    "AY": "OPEN",

    # Liquid
    "L": "LIQUID",
    "R": "LIQUID",

    # Smile / spread
    "EH": "SPREAD",
    "EY": "SPREAD",
    "IH": "SPREAD",
    "IY": "SPREAD",
    "HH": "SPREAD",

    # Tongue / teeth
    "TH": "TH",

    # Closed / narrow
    "T": "NARROW",
    "CH": "NARROW",
    "N": "NARROW",
    "G": "NARROW",
    "S": "NARROW",
    "SH": "NARROW",
    "Z": "NARROW",
    "D": "NARROW",

    # Lip / teeth
    "V": "LIP_TEETH",
    "F": "LIP_TEETH",

    # Neutral / minimal movement
    "B": "NEUTRAL",
    "P": "NEUTRAL",
    "M": "NEUTRAL",
    "W": "NEUTRAL",
    "Y": "NEUTRAL",

    # Rounded / open
    "AO": "ROUND",
    "AW": "ROUND",
    "OW": "ROUND",
    "OY": "ROUND",

    # Tight rounded
    "UH": "TIGHT_ROUND",
    "UW": "TIGHT_ROUND",

    # Silence
    "SIL": "NEUTRAL",
}

In [25]:
# ============================================
# Fast real-time phoneme + viseme inference
# ============================================

import queue
from pathlib import Path

import cv2
import numpy as np
import sounddevice as sd


# ============================================
# Viseme image directory
# ============================================

VISEME_DIR = Path("../src/visemes")


def find_viseme_image(viseme):
    """
    Find the image corresponding to a viseme.

    Works with common image extensions, so you don't
    need to hard-code .png/.jpg/etc.
    """

    for extension in (
        ".png",
        ".jpg",
        ".jpeg",
        ".webp",
        ".bmp",
    ):
        path = VISEME_DIR / f"{viseme}{extension}"

        if path.exists():
            return path

    return None


# ============================================
# Audio state
# ============================================

audio_queue = queue.Queue()

rolling_audio = np.zeros(
    CHUNK_SAMPLES,
    dtype=np.float32
)

block_counter = 0

current_viseme = None
current_viseme_image = None


# ============================================
# Viseme display
# ============================================

WINDOW_NAME = "Viseme"


def update_viseme_display(viseme):
    """
    Update the displayed viseme image.

    The image is only loaded when the viseme changes.
    """

    global current_viseme
    global current_viseme_image

    if viseme == current_viseme:
        return

    current_viseme = viseme

    image_path = find_viseme_image(viseme)

    if image_path is None:
        print(
            f"\nWarning: no image found for viseme "
            f"'{viseme}' in {VISEME_DIR}"
        )

        current_viseme_image = None
        return

    image = cv2.imread(
        str(image_path),
        cv2.IMREAD_COLOR
    )

    if image is None:
        print(
            f"\nWarning: could not load {image_path}"
        )

        current_viseme_image = None
        return

    current_viseme_image = image


# ============================================
# Main real-time loop
# ============================================

try:

    cv2.namedWindow(
        WINDOW_NAME,
        cv2.WINDOW_NORMAL
    )

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
        blocksize=AUDIO_BLOCK_SAMPLES,
        callback=audio_callback,
    ):

        print("Listening...")

        while True:

            new_audio = audio_queue.get()

            # ------------------------------------
            # Update rolling buffer
            # ------------------------------------

            rolling_audio[
                :-len(new_audio)
            ] = rolling_audio[
                len(new_audio):
            ]

            rolling_audio[
                -len(new_audio):
            ] = new_audio

            block_counter += 1

            # ------------------------------------
            # Run inference every 40 ms
            # ------------------------------------

            if (
                block_counter
                % INFERENCE_BLOCKS
                != 0
            ):
                # Still update the OpenCV window
                # so it remains responsive.
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break

                continue

            # ------------------------------------
            # Phoneme inference
            # ------------------------------------

            phonemes, confidences = (
                predict_phonemes(
                    rolling_audio
                )
            )

            # Get newest prediction
            phoneme = phonemes[-1]
            confidence = confidences[-1]

            # ------------------------------------
            # Convert phoneme -> viseme
            # ------------------------------------

            viseme = arpabet_to_visemes.get(
                phoneme,
                "NEUTRAL"
            )

            # ------------------------------------
            # Update viseme image
            # ------------------------------------

            update_viseme_display(viseme)

            # ------------------------------------
            # Update terminal
            # ------------------------------------

            print(
                f"\rPhoneme: {phoneme:<5}"
                f" | Confidence: {confidence:.2f}"
                f" | Viseme: {viseme:<12}",
                end="",
                flush=True
            )

            # ------------------------------------
            # Display current viseme
            # ------------------------------------

            if current_viseme_image is not None:

                cv2.imshow(
                    WINDOW_NAME,
                    current_viseme_image
                )

            # Press Q to quit
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break


except KeyboardInterrupt:

    print("\nStopped.")


finally:

    cv2.destroyAllWindows()

Listening...
Phoneme: [SIL] | Confidence: 0.83 | Viseme: NEUTRAL     
Stopped.
